# ⚙️ [1/4] Check your Google Drive

        DiverScan-release
        │
        ├── data
        │        └── yourdata_folder
        │
        ├── projects
        │        └── yourproject_folder
        │                ├── dataset
        │                ├── optuna
        │                └── training
        │                        ├── epochXX_for_visualization
        │                        │        ├── SummaryTable.xlsx (🟢)
        │                        │        └── ...
        │                        └── ...
        │
        └── utils

Make sure `SummaryTable.xlsx` exists. If it does not, run the last cell of `05_visualize_results.ipynb`, which writes it.


# ⚙️ [2/4] Set up your Gemini API key

This notebook sends the summary table to a Gemini model, so it needs a Google Gemini API key.

- In Colab, open the key icon in the left sidebar (Secrets), add a secret named `GOOGLE_API_KEY` with your key as the value, and allow this notebook to access it.
- Outside Colab, set an environment variable named `GOOGLE_API_KEY` before starting Jupyter.


# ⚙️ [3/4] Authorize this notebook

In [ ]:
# Hit left ▶️ of this cell and run it!
# Mounting Google Drive is what Colab needs to reach your files. Elsewhere
# there is nothing to mount, so the notebook carries on with local paths.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Not running in Google Colab; set the folder paths below to local ones.")


# ⚙️ [4/4] Load the results and ask the model

1.   Fill in the settings below, then select `Runtime` -> `Run all`.

2.   Read the model's interpretation of your attention branches, then ask follow-up questions in the last cell.


In [ ]:
# The Gemini client library. Already installed in Colab; this is a no-op there.
%pip install -q google-genai


In [ ]:
#@title ⚙️ Settings

import os
from pathlib import Path
import time
import pandas as pd
from IPython.display import Markdown, display

project_folder = "/content/drive/MyDrive/DiverScan-release/projects/fruitfly" #@param {type:"string"}
training_folder_name = "training" #@param {type:"string"}
epoch_folder_name = "epoch49_for_visualization" #@param {type:"string"}

#@markdown ---
#@markdown ### `experimental_setting`
#@markdown One phrase describing what was recorded. It is inserted into the question to the model, for example "the courtship behavior of a pair of fruit flies" or "open-field exploration by rats".
experimental_setting = "the courtship behavior of a pair of fruit flies" #@param {type:"string"}

#@markdown ---
#@markdown ### `model_id`
#@markdown The Gemini model to use. You can also type another model name.
model_id = "gemini-3.1-pro-preview" #@param ["gemini-2.5-flash-lite", "gemini-2.5-flash", "gemini-2.5-pro", "gemini-3.1-flash-lite-preview", "gemini-3-flash-preview", "gemini-3.1-pro-preview"] {"allow-input":true}

#@markdown ---
#@markdown ### `reference_documents_folder`
#@markdown Optional. A folder of documents (PDF, Word, Excel, PowerPoint, text, Markdown or JSON) that the model can search while answering, for example papers about your animal. Every file in the folder, including subfolders, is uploaded to Google and indexed.
#@markdown
#@markdown - Give a path such as `/content/drive/MyDrive/DiverScan-release/references/fruitfly` to use the documents.
#@markdown - Leave it empty to chat without any documents.
reference_documents_folder = "/content/drive/MyDrive/DiverScan-release/RAG/fruitfly" #@param {type:"string"}
#@markdown ---

project_folder = Path(project_folder)
training_folder = project_folder / training_folder_name
epoch_visualization_folder = training_folder / epoch_folder_name
summary_table_path = epoch_visualization_folder / "SummaryTable.xlsx"
if not summary_table_path.exists():
    raise FileNotFoundError(
        f"{summary_table_path} was not found. Run the last cell of 05_visualize_results.ipynb first."
    )


def read_settings_file(path):
    values = {}
    with open(path, "r") as f:
        for line in f:
            key, separator, value = line.rstrip("\n").partition(": ")
            if separator:
                values[key] = value
    return values


# The training folder records which optuna folder it was trained from, and
# that folder records the model type and the number of attention branches.
training_settings = read_settings_file(training_folder / "learning_settings.txt")
optuna_folder_name = training_settings.get("optuna_folder_name") or Path(training_settings["optuna_folder"]).name
optuna_settings = read_settings_file(project_folder / optuna_folder_name / "learning_settings.txt")
model_type = optuna_settings["model_type"]
number_of_attention_branches = int(optuna_settings["number_of_attention_branches"])

# Read every sheet of SummaryTable.xlsx as CSV text, so the model can compare
# the attention branches. The "Node attention" sheet exists for multi models only.
sheet_names = [f"Attention{i}" for i in range(1, number_of_attention_branches + 1)]
if model_type == "multi":
    sheet_names.append("Node attention")

summary_table_text = ""
for sheet_name in sheet_names:
    sheet = pd.read_excel(summary_table_path, sheet_name=sheet_name, engine="openpyxl")
    summary_table_text += f"\n===== {sheet_name} =====\n{sheet.to_csv(index=False)}\n"
print(f"Read {len(sheet_names)} sheets from {summary_table_path}")

# The API key comes from the Colab secret, or from the environment elsewhere.
try:
    from google.colab import userdata
    api_key = userdata.get("GOOGLE_API_KEY")
except ImportError:
    api_key = os.environ.get("GOOGLE_API_KEY")
if not api_key:
    raise ValueError("No Gemini API key found. See [2/4] above.")

from google import genai
from google.genai import types

client = genai.Client(api_key=api_key)

# Index the reference documents, if a folder was given, so that the model can
# search them while answering (retrieval-augmented generation).
file_search_store = None
if reference_documents_folder.strip():
    reference_folder = Path(reference_documents_folder)
    if not reference_folder.is_dir():
        raise FileNotFoundError(f"{reference_folder} is not a folder.")

    mime_types_by_suffix = {
        ".pdf": "application/pdf",
        ".docx": "application/vnd.openxmlformats-officedocument.wordprocessingml.document",
        ".doc": "application/msword",
        ".xlsx": "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
        ".xls": "application/vnd.ms-excel",
        ".pptx": "application/vnd.openxmlformats-officedocument.presentationml.presentation",
        ".txt": "text/plain",
        ".md": "text/markdown",
        ".json": "application/json",
    }

    def wait_for_operation(operation, poll_interval_seconds=2):
        while not operation.done:
            time.sleep(poll_interval_seconds)
            operation = client.operations.get(operation)
        return operation

    all_files = [p for p in reference_folder.rglob("*") if p.is_file()]
    supported_files = [p for p in all_files if p.suffix.lower() in mime_types_by_suffix]
    print(f"Found {len(all_files)} files; indexing {len(supported_files)}, "
          f"skipping {len(all_files) - len(supported_files)} with unsupported types.")

    file_search_store = client.file_search_stores.create(
        config={"display_name": f"references-{reference_folder.name}"}
    )
    for document_path in supported_files:
        operation = client.file_search_stores.upload_to_file_search_store(
            file=str(document_path),
            file_search_store_name=file_search_store.name,
            config={
                "display_name": document_path.name,
                "mime_type": mime_types_by_suffix[document_path.suffix.lower()],
                "custom_metadata": [
                    {"key": "path", "string_value": str(document_path.relative_to(reference_folder))},
                    {"key": "ext", "string_value": document_path.suffix.lower()},
                ],
            },
        )
        wait_for_operation(operation)
        print("Indexed:", document_path.name)
else:
    print("No reference documents: the model answers from the summary table alone.")


In [ ]:
#@title 🤖 Start a chat session
#@markdown Run this cell again to start over with a fresh conversation.

# Deterministic settings, so that the same table gives the same answer.
generation_settings = dict(
    temperature=0,
    seed=42,
    candidate_count=1,
    top_p=1.0,
    top_k=1,
)

# Let the model search the reference documents, when they were indexed.
if file_search_store is not None:
    tools = [types.Tool(file_search=types.FileSearch(file_search_store_names=[file_search_store.name]))]
    config = types.GenerateContentConfig(tools=tools, **generation_settings)
else:
    config = types.GenerateContentConfig(**generation_settings)

chat = client.chats.create(model=model_id, config=config)
print("Chat session started with", model_id)


In [ ]:
#@title 🔎 Ask for an interpretation

prompt = f"""
This file shows, for {experimental_setting}, which hand-crafted features had high or low correlation with the attention (from an XAI model).
Compare the characteristics of the {number_of_attention_branches} attention branches and give me an overview.
Also, explain the differences in experimental conditions across each of the attention branches.

Notes on reading the table:
- Features named "... (moving average)" and "... (moving variance)" are the moving average and the moving variance of the base feature over a short window.
- A high value of "Angle from initial location (moving variance)" near the start of a recording does not reflect the angular variance itself; it means that the attention is concentrated near the beginning of the time series.
- Ignore features that simply grow with time (time, Cumulative travel distance, Cumulative change in turning angle), the raw x and y coordinates, Angle from initial location, and Angle from userdefined location.

Table (CSV, one section per sheet):
{summary_table_text}
"""

display(Markdown(chat.send_message(message=prompt).text))


In [ ]:
#@title 💬 Continue the conversation

#@markdown Ask follow-up questions in the same chat session; the model remembers the table and the earlier answers.
#@markdown
#@markdown **How to use**
#@markdown 1. Type your question in `user_message`.
#@markdown 2. Run this cell (hit ▶️ on the left).
#@markdown 3. Change `user_message` and run the cell again to keep the conversation going.
#@markdown
#@markdown **Example questions**
#@markdown - Tell me more about the difference between Attention2 and Attention5.
#@markdown - Which features would be worth examining in a follow-up analysis?
#@markdown - Write Python code that computes the distance between the two animals over time.
#@markdown
#@markdown To start over, run the "Start a chat session" cell again.

user_message = "" #@param {type:"string"}

if user_message.strip():
    display(Markdown(chat.send_message(message=user_message).text))
else:
    print("Type a question in user_message and run the cell.")
